In [1]:
# OpenAI API Examples Notebook
# 
# This notebook demonstrates key features of the OpenAI API for developers.
# It provides practical examples of the most important capabilities with executable code.

# First, let's set up our environment by installing the OpenAI Python library
%pip install openai numpy matplotlib pydantic --upgrade --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Import the OpenAI library
import os
from openai import OpenAI

In [5]:
MODEL = "gpt-4.1-mini"

In [6]:

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY_PRACTICE")
    # api_key="YOUR_API_KEY_HERE"
)

## 1. Basic Text Generation

In [5]:
prompt = "Write a one-sentence bedtime story about a unicorn."

response = client.responses.create(
    model=MODEL,
    input=prompt
)

print(f"Prompt: {prompt}")
print(f"Response: {response.output_text}")

# Show how to control generation with parameters
print("\nWith controlled parameters:")
response = client.responses.create(
    model=MODEL,
    input=prompt,
    temperature=0.7,  # Lower for more deterministic outputs
    top_p=0.9         # Nucleus sampling
)

print(f"Response (controlled): {response.output_text}")

Prompt: Write a one-sentence bedtime story about a unicorn.
Response: Under a shimmering moon, a gentle unicorn painted the night sky with sparkling stars, whispering sweet dreams to all who sleep.

With controlled parameters:
Response (controlled): Under a sky sprinkled with shimmering stars, a gentle unicorn named Luna whispered dreams of magic and kindness to every child as they drifted peacefully to sleep.


---

## 2. Structured Outputs

In [ ]:
import json

#New Need a more easy example
response = client.responses.create(
   model=MODEL,
    input=[
        {"role": "system", "content": "You are a UI generator AI. Convert the user input into a UI."},
        {"role": "user", "content": "Make a User Profile Form"}
    ],
    text={
        "format": {
            "type": "json_schema",
            "name": "ui",
            "description": "Dynamically generated UI",
            "schema": {
                "type": "object",
                "properties": {
                    "type": {
                        "type": "string",
                        "description": "The type of the UI component",
                        "enum": ["div", "button", "header", "section", "field", "form"]
                    },
                    "label": {
                        "type": "string",
                        "description": "The label of the UI component, used for buttons or form fields"
                    },
                    "children": {
                        "type": "array",
                        "description": "Nested UI components",
                        "items": {"$ref": "#"}
                    },
                    "attributes": {
                        "type": "array",
                        "description": "Arbitrary attributes for the UI component, suitable for any element",
                        "items": {
                            "type": "object",
                            "properties": {
                              "name": {
                                  "type": "string",
                                  "description": "The name of the attribute, for example onClick or className"
                              },
                              "value": {
                                  "type": "string",
                                  "description": "The value of the attribute"
                              }
                          },
                          "required": ["name", "value"],
                          "additionalProperties": False
                      }
                    }
                },
                "required": ["type", "label", "children", "attributes"],
                "additionalProperties": False
            },
            "strict": True,
        },
    },
)

ui = json.loads(response.output_text)


In [7]:
ui

{'type': 'form',
 'label': 'User Profile Form',
 'children': [{'type': 'field',
   'label': 'First Name',
   'children': [],
   'attributes': [{'name': 'type', 'value': 'text'},
    {'name': 'name', 'value': 'firstName'},
    {'name': 'placeholder', 'value': 'Enter your first name'}]},
  {'type': 'field',
   'label': 'Last Name',
   'children': [],
   'attributes': [{'name': 'type', 'value': 'text'},
    {'name': 'name', 'value': 'lastName'},
    {'name': 'placeholder', 'value': 'Enter your last name'}]},
  {'type': 'field',
   'label': 'Email',
   'children': [],
   'attributes': [{'name': 'type', 'value': 'email'},
    {'name': 'name', 'value': 'email'},
    {'name': 'placeholder', 'value': 'Enter your email'}]},
  {'type': 'field',
   'label': 'Password',
   'children': [],
   'attributes': [{'name': 'type', 'value': 'password'},
    {'name': 'name', 'value': 'password'},
    {'name': 'placeholder', 'value': 'Enter your password'}]},
  {'type': 'field',
   'label': 'Date of Birth',


In [19]:
import json

response = client.responses.create(
    model=MODEL,
    input=(
        "Generate a list of Indian Prime Ministers. "
        "For each, write a light, respectful, satirical humorous profile highlighting their mistakes and problems. "
        "Each profile must be 20 to 30 words."
    ),
    text={
        "format": {
            "type": "json_schema",
            "name": "jokers_list",   # ✅ REQUIRED without this it will error
            "schema": {
                "type": "object",
                "properties": {
                    "joker_list": {     # *same field should be used in required field below
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "name": { "type": "string" },
                                "profile": { "type": "string" }
                            },
                            "required": ["name", "profile"],
                            "additionalProperties": False
                        }
                    }
                },
                "required": ["joker_list"], # *same field as above marked with *
                "additionalProperties": False
            },
            "strict": True
        }
    }
)


if not response.output_text:
    raise RuntimeError("Model did not produce structured output")

data = json.loads(response.output_text)
print(json.dumps(data, indent=2))



{
  "joker_list": [
    {
      "name": "Jawaharlal Nehru",
      "profile": "Pioneered India's independence, occasionally mixed socialism with idealism \u2014 left a legacy as complex as his famous rose garden, where sometimes the thorns got overlooked."
    },
    {
      "name": "Lal Bahadur Shastri",
      "profile": "Remembered for 'Jai Jawan Jai Kisan,' but sometimes his quiet demeanor masked indecisive moments; a leader less loud, more thoughtful, quietly steering tricky waters."
    },
    {
      "name": "Indira Gandhi",
      "profile": "Bold and decisive, yet her emergency period suggested power naps could be longer; her iron will balanced between visionary leadership and authoritarian streaks."
    },
    {
      "name": "Morarji Desai",
      "profile": "Known for honesty and cane, but his old-school style sometimes slowed progress\u2014proving that discipline is key, but speed matters too in politics."
    },
    {
      "name": "Charan Singh",
      "profile": "Champion 

---

## 2.1 Structured Outputs with Pydantic models

In [8]:
from pydantic import BaseModel

class Step(BaseModel):
    explanation: str
    output: str

class MathReasoning(BaseModel):
    steps: list[Step]
    final_answer: str

completion = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful math tutor. Guide the user through the solution step by step."},
        {"role": "user", "content": "how can I solve 8x + 7 = -23"}
    ],
    response_format=MathReasoning,
)

math_reasoning = completion.choices[0].message

# If the model refuses to respond, you will get a refusal message
if (math_reasoning.refusal):
    print(math_reasoning.refusal)
else:
    print(math_reasoning.parsed)

steps=[Step(explanation='Start with the equation 8x + 7 = -23. To isolate the term with x, subtract 7 from both sides.', output='8x = -23 - 7'), Step(explanation='Simplify the right side by performing the subtraction: -23 - 7 = -30.', output='8x = -30'), Step(explanation='To solve for x, divide both sides of the equation by 8.', output='x = \\frac{-30}{8}'), Step(explanation='Simplify the fraction by dividing numerator and denominator by 2.', output='x = \\frac{-15}{4} = -3.75')] final_answer='x = -\\frac{15}{4} or -3.75'


In [9]:
math_reasoning.parsed.steps[0]

Step(explanation='Start with the equation 8x + 7 = -23. To isolate the term with x, subtract 7 from both sides.', output='8x = -23 - 7')

---

## 3. Multimodal Capabilities - Vision

In [10]:
# For notebook demonstration, we'll use a placeholder URL
image_url = "https://images.unsplash.com/photo-1579546929518-9e396f3cc809?ixlib=rb-4.0.3&ixid=MnwxMjA3fDB8MHxleHBsb3JlLWZlZWR8MXx8fGVufDB8fHx8&w=1000&q=80"

response = client.responses.create(
    model=MODEL,
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text", "text": "what's in this image?"},
            {
                "type": "input_image",
                "image_url": image_url,
            },
        ],
    }],
)

print(response.output_text)


This image shows a colorful gradient blend with smooth transitions between shades of pink, red, orange, yellow, teal, and blue. There are no distinct objects or shapes, just a soft, abstract mix of colors.


---

## 4. Audio Capabilities - Text to Speech

In [11]:
speech_file_path = "speech.mp3"

with client.audio.speech.with_streaming_response.create(
    model="gpt-4o-mini-tts",
    voice="coral",
    input="Today is a wonderful day to build something people love!",
    instructions="Speak in a cheerful and positive tone.",
) as response:
    response.stream_to_file(speech_file_path)

---

## 4.1 Audio Capabilities - Speech to Text

In [12]:
audio_file= open("speech.mp3", "rb")

transcription = client.audio.transcriptions.create(
    model="gpt-4o-transcribe", 
    file=audio_file
)

print(transcription.text)

Today is a wonderful day to build something people love.


---

## 5. Function Calling

In [24]:
import requests

def get_weather(latitude, longitude):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    return data['current']['temperature_2m']

tools = [{
    "type": "function",
    "name": "get_weather",
    "description": "Get current temperature for provided coordinates in celsius.",
    "parameters": {
        "type": "object",
        "properties": {
            "latitude": {"type": "number"},
            "longitude": {"type": "number"}
        },
        "required": ["latitude", "longitude"],
        "additionalProperties": False
    },
    "strict": True
}]

input_messages = [{"role": "user", "content": "What's the weather like in Paris today?"}]

response = client.responses.create(
    model="gpt-4.1-mini",
    input=input_messages,
    tools=tools,
)

In [25]:
# Extract the tool call and arguments
from pprint import pprint

tool_call = response.output[0]
print("Tool call from response:")
pprint(tool_call, indent=2, width=100)
print("\n" + "="*60 + "\n")
print(tool_call)
args = json.loads(tool_call.arguments)
# Call the function
result = get_weather(args["latitude"], args["longitude"])
print(f"Current temperature in Paris: {result}°C")

Tool call from response:
ResponseFunctionToolCall(arguments='{"latitude":48.8566,"longitude":2.3522}', call_id='call_zh6BnqFMeGLyeg8ldWImWs9n', name='get_weather', type='function_call', id='fc_014369b6b262610000695e8f5aec948193be2ea6a455619a95', status='completed')


ResponseFunctionToolCall(arguments='{"latitude":48.8566,"longitude":2.3522}', call_id='call_zh6BnqFMeGLyeg8ldWImWs9n', name='get_weather', type='function_call', id='fc_014369b6b262610000695e8f5aec948193be2ea6a455619a95', status='completed')
Current temperature in Paris: 2.3°C


In [26]:
# Append the tool call and result to the input messages
input_messages.append(tool_call) # append model's function call message
input_messages.append({ # append result message
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": str(result)
})

response_2 = client.responses.create(
    model="gpt-4.1-mini",
    input=input_messages,
    tools=tools,
)
print(response_2.output_text)

The weather in Paris today is quite cool with a temperature of around 2.3 degrees Celsius. Would you like to know more details, such as the weather forecast or any specific information?


## Last 5 years temperature 
OpenAI does not have a built-in weather tool. You must expose your own tool that calls a weather historical/archive API, then let the model call it.

In [7]:
#No open API call example

import requests
from datetime import date

LATITUDE = 29.3803
LONGITUDE = 79.4636

def fetch_feb_temperature(year):
    start_date = f"{year}-02-01"
    end_date = f"{year}-02-06"

    url = (
        "https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={LATITUDE}"
        f"&longitude={LONGITUDE}"
        f"&start_date={start_date}"
        f"&end_date={end_date}"
        "&daily=temperature_2m_max,temperature_2m_min"
        "&timezone=Asia/Kolkata"
    )

    response = requests.get(url)
    response.raise_for_status()
    return response.json()

def get_last_5_years_feb_data():
    current_year = date.today().year
    results = {}

    for year in range(current_year - 5, current_year):
        data = fetch_feb_temperature(year)
        results[year] = data["daily"]

    return results

if __name__ == "__main__":
    weather_data = get_last_5_years_feb_data()

    for year, daily in weather_data.items():
        print(f"\nYear: {year}")
        for d, tmin, tmax in zip(
            daily["time"],
            daily["temperature_2m_min"],
            daily["temperature_2m_max"]
        ):
            print(f"{d} | Min: {tmin}°C | Max: {tmax}°C")



Year: 2021
2021-02-01 | Min: 0.3°C | Max: 12.9°C
2021-02-02 | Min: 1.7°C | Max: 14.8°C
2021-02-03 | Min: 1.1°C | Max: 13.2°C
2021-02-04 | Min: 0.5°C | Max: 11.1°C
2021-02-05 | Min: 0.6°C | Max: 7.5°C
2021-02-06 | Min: -1.0°C | Max: 10.2°C

Year: 2022
2022-02-01 | Min: -0.9°C | Max: 8.7°C
2022-02-02 | Min: 0.4°C | Max: 10.3°C
2022-02-03 | Min: 0.6°C | Max: 4.6°C
2022-02-04 | Min: -1.6°C | Max: 3.6°C
2022-02-05 | Min: -4.0°C | Max: 5.2°C
2022-02-06 | Min: -1.7°C | Max: 9.8°C

Year: 2023
2023-02-01 | Min: 5.0°C | Max: 16.1°C
2023-02-02 | Min: 5.0°C | Max: 16.5°C
2023-02-03 | Min: 4.6°C | Max: 14.3°C
2023-02-04 | Min: 4.7°C | Max: 15.7°C
2023-02-05 | Min: 2.5°C | Max: 14.0°C
2023-02-06 | Min: 2.6°C | Max: 13.4°C

Year: 2024
2024-02-01 | Min: 0.6°C | Max: 5.9°C
2024-02-02 | Min: -1.2°C | Max: 9.4°C
2024-02-03 | Min: -0.2°C | Max: 12.8°C
2024-02-04 | Min: 2.2°C | Max: 9.2°C
2024-02-05 | Min: 0.1°C | Max: 9.2°C
2024-02-06 | Min: -1.3°C | Max: 8.8°C

Year: 2025
2025-02-01 | Min: 1.3°C | Max: 

Open AI wrapper on the custom tool

In [14]:
import requests
import json
from datetime import date
from pprint import pprint

def historical_temperature(latitude: float, longitude: float, last_n_years: int):
    current_year = date.today().year
    results = {}

    for year in range(current_year - last_n_years, current_year):
        start_date = f"{year}-02-01"
        end_date = f"{year}-02-06"

        url = (
            "https://archive-api.open-meteo.com/v1/archive"
            f"?latitude={latitude}"
            f"&longitude={longitude}"
            f"&start_date={start_date}"
            f"&end_date={end_date}"
            "&daily=temperature_2m_min,temperature_2m_max"
            "&timezone=Asia/Kolkata"
        )

        r = requests.get(url)
        r.raise_for_status()
        results[year] = r.json()["daily"]

    return results


tools = [
    {
        "type": "function",
        "name": "historical_temperature",
        "description": "Get daily min and max temperatures for a location from Feb 1 to Feb 6 for the last N years.",
        "parameters": {
            "type": "object",
            "properties": {
                "latitude": {"type": "number"},
                "longitude": {"type": "number"},
                "last_n_years": {"type": "integer"}
            },
            "required": ["latitude", "longitude", "last_n_years"],
            "additionalProperties": False
        }
    }
]

# 1️⃣ Ask model (force tool call)
response = client.responses.create(
    model="gpt-4.1-mini",
    input="Show me Nainital temperatures from Feb 1 to Feb 6 for the last 5 years.",
    tools=tools,
    tool_choice="required"
)

pprint(response.output)

# 2️⃣ Handle tool call EXACTLY ONCE
for item in response.output:
    if item.type in ("tool_call", "function_call"):
        args = json.loads(item.arguments)  # ✅ FIX
        tool_data = historical_temperature(**args)

        followup = client.responses.create(
            model="gpt-4.1-mini",
            previous_response_id=response.id,
            input=[
                {
                    "type": "function_call_output",
                    "call_id": item.call_id,
                    "output": json.dumps(tool_data)
                }
            ]
        )

        print("\n=== FINAL ANSWER ===\n")
        print(followup.output_text)



[ResponseFunctionToolCall(arguments='{"latitude":29.3842,"longitude":79.4542,"last_n_years":5}', call_id='call_y8aMoimxGrhI8uqChHl9x9qh', name='historical_temperature', type='function_call', id='fc_0afc404faf4b533a00695f0ae6e0488193a43b184b08030b14', status='completed')]

=== FINAL ANSWER ===

Here are the temperatures in Nainital from February 1 to February 6 for the last 5 years:

2021:
- Min (°C): -1.1, 0.3, -0.3, -0.9, -0.8, -2.4
- Max (°C): 11.5, 13.4, 11.8, 9.7, 6.1, 8.8

2022:
- Min (°C): -2.3, -1.0, -0.8, -3.0, -5.4, -3.1
- Max (°C): 7.3, 8.9, 3.2, 2.2, 3.8, 8.4

2023:
- Min (°C): 3.6, 3.6, 3.2, 3.3, 1.1, 1.2
- Max (°C): 14.7, 15.1, 12.9, 14.3, 12.6, 12.0

2024:
- Min (°C): -0.8, -2.6, -1.6, 0.8, -1.3, -2.7
- Max (°C): 4.5, 8.0, 11.4, 7.8, 7.8, 7.4

2025:
- Min (°C): -0.1, -0.1, 0.9, 2.5, -0.4, -2.6
- Max (°C): 12.1, 12.6, 16.1, 10.6, 11.2, 12.8

If you need a specific format or additional data, please let me know!


---

## 6. Reasoning Models

In [16]:
# response_o1 = client.responses.create(
#     model="o1",
#     input="Design an algorithm to find the shortest path in a graph."
# )

# print("o1 Response:")
# print(response_o1.output_text)

# Using o4-mini for faster reasoning
response_o4 = client.responses.create(
    model="o4-mini",
    input="Explain how to implement a hash table."
)

print("\no3-mini Response:")
print(response_o4.output_text)


o3-mini Response:
A hash table (or hash map) is a data structure that provides (on average) O(1) time for insertion, lookup, and deletion by mapping keys to indices in an underlying array via a hash function. A basic implementation involves these steps:

1. Choose the Bucket Array  
   • Create an array (often called “buckets”) of some initial size N (e.g. 16).  
   • Each bucket will hold zero or more key-value pairs.

2. Define a Hash Function  
   • The hash function h(key) should map a key to a non-negative integer.  
   • To get an index in [0…N–1], compute:  
     index = h(key) mod N  
   • Good hash functions distribute keys uniformly.

3. Collision Resolution  
   Since multiple keys can map to the same index, you need a collision strategy. Two common ones are:  
   a. Separate Chaining  
     – Each bucket contains a linked list (or dynamic array) of entries.  
     – On insert, append the (key,value) to the list if the key isn’t already there.  
     – On lookup or delete, 

In [17]:
## With reasoning models, you can view the reasoning process and the steps taken to arrive at the answer.
print("This is the reasoning configuration for o4-mini:", response_o4.reasoning)

This is the reasoning configuration for o4-mini: Reasoning(effort='medium', generate_summary=None, summary=None)


## 7. Embeddings

In [18]:
import numpy as np

# First, let's create embeddings for a set of words
words = [
    "king", "queen", "man", "woman", 
    "apple", "banana", "orange", "pear",
    "castle", "throne"
]

# Get embeddings for all words
response = client.embeddings.create(
    model="text-embedding-3-large",
    input=words,
    encoding_format="float"
)

# Extract the embeddings
embeddings = [data.embedding for data in response.data]

print(f"Embedding dimension: {len(embeddings[0])}")
print(f"Number of embeddings: {len(embeddings)}")

# Function to compute dot product between two vectors
def dot_product(vec1, vec2):
    return np.dot(vec1, vec2)

# Compute similarity matrix (dot products between all pairs)
similarity_matrix = np.zeros((len(words), len(words)))
for i in range(len(words)):
    for j in range(len(words)):
        similarity_matrix[i][j] = dot_product(embeddings[i], embeddings[j])

# Print similarity matrix with labels
print("\nSimilarity Matrix (Dot Products):")
print("          " + " ".join(f"{word:<8}" for word in words))
for i, word in enumerate(words):
    row_values = " ".join(f"{similarity_matrix[i][j]:.4f}  " for j in range(len(words)))
    print(f"{word:<10} {row_values}")

# Check specific relationships
king_queen_similarity = dot_product(embeddings[0], embeddings[1])
apple_banana_similarity = dot_product(embeddings[4], embeddings[5])

print("\nSpecific relationships:")
print(f"Similarity between 'king' and 'queen': {king_queen_similarity:.4f}")
print(f"Similarity between 'apple' and 'banana': {apple_banana_similarity:.4f}")
print(f"Similarity between 'king' and 'apple': {dot_product(embeddings[0], embeddings[4]):.4f}")

# We expect king/queen to be closer to each other than king/apple
# And apple/banana to be closer to each other than queen/banana

Embedding dimension: 3072
Number of embeddings: 10

Similarity Matrix (Dot Products):
          king     queen    man      woman    apple    banana   orange   pear     castle   throne  
king       1.0000   0.5552   0.4183   0.2938   0.3243   0.3305   0.2879   0.2802   0.3615   0.4027  
queen      0.5552   1.0000   0.3072   0.4132   0.3145   0.3191   0.2983   0.2996   0.2969   0.3354  
man        0.4183   0.3072   1.0000   0.5713   0.3098   0.3495   0.2972   0.2695   0.2998   0.2650  
woman      0.2938   0.4132   0.5713   1.0000   0.3199   0.2937   0.2784   0.2533   0.2449   0.2491  
apple      0.3243   0.3145   0.3098   0.3199   1.0000   0.4619   0.4588   0.4391   0.3002   0.2340  
banana     0.3305   0.3191   0.3495   0.2937   0.4619   1.0000   0.4579   0.3636   0.2777   0.2075  
orange     0.2879   0.2983   0.2972   0.2784   0.4588   0.4579   1.0000   0.3822   0.2848   0.2174  
pear       0.2802   0.2996   0.2695   0.2533   0.4391   0.3636   0.3822   1.0000   0.2788   0.1929  
castle